In [1]:
pip install cooper-optim


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import torch
import torch.nn as nn
import cooper
from cooper.penalty_coefficients import MultiplicativePenaltyCoefficientUpdater

In [3]:
def compute_d(arch):
    return sum(arch[i] * arch[i + 1] for i in range(len(arch) - 1))

In [4]:
def unpack_weights(w_flat, arch):
    mats = []
    idx = 0
    for i in range(len(arch) - 1):
        n_in, n_out = arch[i], arch[i + 1]
        size = n_in * n_out
        mats.append(w_flat[idx:idx + size].reshape(n_out, n_in))
        idx += size
    return mats

In [5]:
def forward_pass(u, w_flat, biases, arch):
    weight_mats = unpack_weights(w_flat, arch)
    a = u
    for l, W in enumerate(weight_mats):
        z = W @ a + biases[l].unsqueeze(1)
        a = torch.tanh(z) if l < len(weight_mats) - 1 else z
    return a

In [6]:
def binary_entropy(Q, eps=1e-6):
    Qc = Q.clamp(eps, 1.0 - eps)
    return -(Qc * Qc.log() + (1.0 - Qc) * (1.0 - Qc).log()).sum()

In [7]:
class SparseNNCMP(cooper.ConstrainedMinimizationProblem):
    def __init__(
        self,
        model,
        c0,
        u_data,
        y_data,
        ent_tol=1e-1,
        col_tol=1e-3,
        init_penalty_entropy=1.0,
        init_penalty_colsum=5.0,
    ):
        super().__init__()
        self.model = model
        self.c0 = c0
        self.u_data = u_data
        self.y_data = y_data
        self.ent_tol = ent_tol
        self.col_tol = col_tol

        k = model.k

        self.entropy_upper = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=1),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.tensor([float(init_penalty_entropy)], dtype=torch.float32)
            ),
        )

        self.entropy_lower = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=1),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.tensor([float(init_penalty_entropy)], dtype=torch.float32)
            ),
        )

        self.colsum_upper = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=k),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.full((k,), float(init_penalty_colsum), dtype=torch.float32)
            ),
        )

        self.colsum_lower = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=k),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.full((k,), float(init_penalty_colsum), dtype=torch.float32)
            ),
        )
    def compute_cmp_state(self):
        m = self.u_data.shape[1]
        y_pred = self.model(self.u_data)
        loss = ((self.y_data - y_pred) ** 2).sum() / m

        H = binary_entropy(self.model.Q, eps=self.model.eps)
        ent_upper_violation = (H - self.c0 - self.ent_tol).reshape(1)
        ent_lower_violation = (self.c0 - H - self.ent_tol).reshape(1)

        colsum = self.model.Q.sum(dim=0)
        col_upper_violation = colsum - 1.0 - self.col_tol
        col_lower_violation = 1.0 - colsum - self.col_tol

        return cooper.CMPState(
            loss=loss,
            observed_constraints={
                self.entropy_upper: cooper.ConstraintState(violation=ent_upper_violation),
                self.entropy_lower: cooper.ConstraintState(violation=ent_lower_violation),
                self.colsum_upper: cooper.ConstraintState(violation=col_upper_violation),
                self.colsum_lower: cooper.ConstraintState(violation=col_lower_violation),
            },
        )


In [ ]:
def run_homotopy(
    arch,
    u_data,
    y_data,
    k,
    c0_start,
    c0_end=1.0,
    beta=0.95,
    outer_iters=80,
    inner_iters=200,
    primal_lr=1e-3,
    dual_lr=1e-2,
    eps=1e-6,
    ent_tol=1e-1,
    col_tol=1e-3,
    init_penalty_entropy=1.0,
    init_penalty_colsum=5.0,
    penalty_growth=1.2,
    max_penalty_entropy=50.0,
    max_penalty_colsum=500.0,
    log_every=10,
):
    torch.manual_seed(42)

    d = compute_d(arch)
    print(f"Architecture: {arch}, d={d}, k={k}")
    print(f"c0_start={c0_start:.4f}, c0_end={c0_end:.4f}, beta={beta}, outer={outer_iters}, inner={inner_iters}")
    print(f"primal_lr={primal_lr}, dual_lr={dual_lr}")
    print(f"init_penalty_entropy={init_penalty_entropy}, init_penalty_colsum={init_penalty_colsum}")

    model = SparseNNCMP(arch, k, eps=eps)
    H0 = binary_entropy(model.Q, eps).item()
    print(f"Initial entropy H(Q) = {H0:.4f}")
    print(f"Initial colsum error = {(model.Q.sum(dim=0) - 1.0).abs().max().item():.4e}")

    cmp = SparseNNCMP(
        model=model,
        c0=c0_start,
        u_data=u_data,
        y_data=y_data,
        ent_tol=ent_tol,
        col_tol=col_tol,
        init_penalty_entropy=init_penalty_entropy,
        init_penalty_colsum=init_penalty_colsum,
    )

    primal_opt = torch.optim.Adam(model.parameters(), lr=primal_lr)
    dual_opt = torch.optim.SGD(cmp.dual_parameters(), lr=dual_lr, maximize=True)

    cooper_opt = cooper.optim.AlternatingPrimalDualOptimizer(
        cmp=cmp,
        primal_optimizers=primal_opt,
        dual_optimizers=dual_opt,
    )

    entropy_updater = MultiplicativePenaltyCoefficientUpdater(
        growth_factor=penalty_growth,
        violation_tolerance=ent_tol,
        has_restart=True,
    )
    colsum_updater = MultiplicativePenaltyCoefficientUpdater(
        growth_factor=penalty_growth,
        violation_tolerance=col_tol,
        has_restart=False,
    )

    history = []
    c0 = c0_start

    for outer in range(outer_iters):
        cmp.c0 = c0

        roll_out = None
        for inner in range(inner_iters):
            roll_out = cooper_opt.roll()

            if (not torch.isfinite(model.Q).all()) or (not torch.isfinite(model.x_coeff).all()):
                print(f"NaN/Inf detected at outer={outer}, inner={inner}")
                return model, history

        model.clamp_Q()

        obs = roll_out.cmp_state.observed_constraints

        entropy_updater.step({
            cmp.entropy_upper: obs[cmp.entropy_upper],
            cmp.entropy_lower: obs[cmp.entropy_lower],
        })
        colsum_updater.step({
            cmp.colsum_upper: obs[cmp.colsum_upper],
            cmp.colsum_lower: obs[cmp.colsum_lower],
        })

        with torch.no_grad():
            cmp.entropy_upper.penalty_coefficient.value.clamp_(max=max_penalty_entropy)
            cmp.entropy_lower.penalty_coefficient.value.clamp_(max=max_penalty_entropy)
            cmp.colsum_upper.penalty_coefficient.value.clamp_(max=max_penalty_colsum)
            cmp.colsum_lower.penalty_coefficient.value.clamp_(max=max_penalty_colsum)

            Q_now = model.Q.detach()
            H_val = binary_entropy(Q_now, eps).item()
            colsum = Q_now.sum(dim=0)
            colsum_err = (colsum - 1.0).abs().max().item()
            y_pred = model(u_data)
            loss_val = ((y_data - y_pred) ** 2).sum().item() / u_data.shape[1]
            gap = torch.minimum(Q_now, 1.0 - Q_now).max().item()

            cEu = cmp.entropy_upper.penalty_coefficient.value.max().item()
            cEl = cmp.entropy_lower.penalty_coefficient.value.max().item()
            cU = cmp.colsum_upper.penalty_coefficient.value.max().item()
            cL = cmp.colsum_lower.penalty_coefficient.value.max().item()

            lamEu = cmp.entropy_upper.multiplier.weight.max().item()
            lamEl = cmp.entropy_lower.multiplier.weight.max().item()
            lamU = cmp.colsum_upper.multiplier.weight.max().item()
            lamL = cmp.colsum_lower.multiplier.weight.max().item()



        history.append(
            {
                "outer": outer,
                "c0": c0,
                "H": H_val,
                "loss": loss_val,
                "gap": gap,
                "colsum_err": colsum_err,
                "cEu": cEu,
                "cEl": cEl,
                "cU": cU,
                "cL": cL,
                "lamEu": lamEu,
                "lamEl": lamEl,
                "lamU": lamU,
                "lamL": lamL,
            }
        )

        if outer % log_every == 0 or outer == outer_iters - 1:
            print(
                f"[{outer:3d}] c0={c0:.4f}  H={H_val:.3f}  loss={loss_val:.3e}  "
                f"gap={gap:.3e}  cs_err={colsum_err:.2e}  "
                f"pen(Eu/El/U/L)=({cEu:.2f}/{cEl:.2f}/{cU:.2f}/{cL:.2f})  "
                f"mult(Eu/El/U/L)=({lamEu:.2e}/{lamEl:.2e}/{lamU:.2e}/{lamL:.2e})"
            )

        c0= beta*c0

    return model, history


def main():
    np.random.seed(42)

    m = 500
    u_np = np.random.uniform(-2, 2, size=(1, m))
    y_np = u_np ** 3 + u_np ** 2 + 1.0 + 0.1 * np.random.randn(1, m)

    u_data = torch.tensor(u_np, dtype=torch.float32)
    y_data = torch.tensor(y_np, dtype=torch.float32)

    arch = [1, 10, 10, 1]
    k = 20
    d = compute_d(arch)
    print(f"d = {d} (expected 120)")

    tmp = SparseNNCMP(arch, k)
    c0_start = binary_entropy(tmp.Q).item()
    del tmp
    print(f"Starting c0 = {c0_start:.4f}")

    model, history = run_homotopy(
        arch=arch,
        u_data=u_data,
        y_data=y_data,
        k=k,
        c0_start=c0_start,
        c0_end=1.0,
        beta=0.95,
        outer_iters=80,
        inner_iters=200,
        primal_lr=1e-3,
        dual_lr=1e-2,
        eps=1e-6,
        ent_tol=1e-1,
        col_tol=1e-3,
        init_penalty_entropy=1.0,
        init_penalty_colsum=5.0,
        penalty_growth=1.2,
        max_penalty_entropy=50.0,
        max_penalty_colsum=500.0,
        log_every=10,
    )

    Q_final = model.Q.detach().cpu().numpy()
    x_final = model.x_coeff.detach().cpu().numpy()
    w_final = Q_final @ x_final

    print("\n" + "=" * 60)
    print("FINAL RESULTS")
    print("=" * 60)
    if history:
        print(f"Final Q binary gap (max min(q,1-q)): {np.max(np.minimum(Q_final, 1.0 - Q_final)):.6f}")
        print(f"Final colsum error: {np.max(np.abs(Q_final.sum(axis=0) - 1.0)):.6f}")
        print(f"Final entropy H(Q): {binary_entropy(model.Q).item():.6f}")
        print(f"Final loss: {history[-1]['loss']:.6f}")
        print(f"w has {np.sum(np.abs(w_final) > 1e-6)} nonzeros out of {len(w_final)}")

        gaps = np.minimum(Q_final, 1.0 - Q_final)
        print("\nQ entry distribution:")
        print(f"  entries with gap < 1e-4  (nearly binary): {(gaps < 1e-4).sum()} / {Q_final.size}")
        print(f"  entries with gap < 1e-2:                   {(gaps < 1e-2).sum()} / {Q_final.size}")
        print(f"  entries with gap > 0.1   (ambiguous):     {(gaps > 0.1).sum()} / {Q_final.size}")
        print(f"  entries with gap > 0.4   (near 0.5):      {(gaps > 0.4).sum()} / {Q_final.size}")

        print("\nEntries of Q > 0.9:")
        rows, cols = np.where(Q_final > 0.9)
        for r, c in zip(rows, cols):
            print(f"  Q[{r},{c}] = {Q_final[r, c]:.6f}  ->  w[{r}] = {w_final[r]:.4f}")


if __name__ == "__main__":
    main()

d = 120 (expected 120)


NameError: name 'SparseNNModel' is not defined